<a href="https://colab.research.google.com/github/forlslandbis-a11y/Wyvern-Chess-engine/blob/main/Wyvern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 구글 드라이브 마운트 (ONNX 모델 및 체크포인트 자동 저장용)
from google.colab import drive
import os, sys

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/Wyvern_Engine'
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. GitHub 리포지토리 클론 (forlslandbis-a11y 계정 적용)
# ※ 만약 GitHub의 실제 저장소 이름이 wyvern-chess-engine이 아니라면 끝부분 이름만 맞춰주세요!
GITHUB_REPO_URL = "https://github.com/forlslandbis-a11y/wyvern-chess-engine.git"
REPO_NAME = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')

if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO_URL}

%cd {REPO_NAME}

# 3. 필수 라이브러리 설치
!pip install -q python-chess torch onnx onnxruntime numpy

Mounted at /content/drive
Cloning into 'wyvern-chess-engine'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 49 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 11.38 KiB | 5.69 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/wyvern-chess-engine
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 67.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 93.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import chess
import numpy as np

# 체스판 64칸을 12채널(6기물 x 2색상) 텐서로 변환
def board_to_tensor(board: chess.Board) -> torch.Tensor:
    tensor = np.zeros((12, 8, 8), dtype=np.float32)
    piece_map = {
        chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
        chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5
    }
    for square, piece in board.piece_map().items():
        row = 7 - (square // 8)
        col = square % 8
        channel = piece_map[piece.piece_type] + (0 if piece.color == chess.WHITE else 6)
        tensor[channel, row, col] = 1.0

    if board.turn == chess.BLACK:
        tensor = np.flip(tensor, axis=(1, 2)).copy()
    return torch.from_numpy(tensor)

# Residual Block
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)

# Wyvern Net Main
class WyvernNet(nn.Module):
    def __init__(self, num_blocks=8, num_channels=256):
        super().__init__()
        self.start_conv = nn.Sequential(
            nn.Conv2d(12, num_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(num_channels),
            nn.ReLU()
        )
        self.res_blocks = nn.ModuleList([ResBlock(num_channels) for _ in range(num_blocks)])

        # Policy Head (어느 수가 좋은지 확률 반환)
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_channels, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 1968)
        )
        # Value Head (현재 국면 승률 -1.0 ~ +1.0 반환)
        self.value_head = nn.Sequential(
            nn.Conv2d(num_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.start_conv(x)
        for block in self.res_blocks:
            x = block(x)
        return self.policy_head(x), self.value_head(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WyvernNet().to(device)
print(f"🐉 WyvernNet 로드 완료 (사용 가속기: {device})")

🐉 WyvernNet 로드 완료 (사용 가속기: cuda)


In [3]:
import random

def move_to_index(move: chess.Move) -> int:
    return (move.from_square * 64 + move.to_square) % 1968

def self_play_game(model, device, max_moves=100):
    board = chess.Board()
    states, policies, values = [], [], []

    model.eval()
    while not board.is_game_over() and len(states) < max_moves:
        state_tensor = board_to_tensor(board).unsqueeze(0).to(device)

        with torch.no_grad():
            policy_logits, _ = model(state_tensor)

        legal_moves = list(board.legal_moves)
        if not legal_moves:
            break

        legal_indices = [move_to_index(m) for m in legal_moves]
        probs = F.softmax(policy_logits[0, legal_indices], dim=-1).cpu().numpy()

        chosen_move = random.choices(legal_moves, weights=probs)[0]

        target_policy = np.zeros(1968, dtype=np.float32)
        target_policy[move_to_index(chosen_move)] = 1.0

        states.append(board_to_tensor(board))
        policies.append(target_policy)

        board.push(chosen_move)

    result = board.result()
    winner = 1.0 if result == "1-0" else (-1.0 if result == "0-1" else 0.0)

    for i in range(len(states)):
        values.append([winner if i % 2 == 0 else -winner])

    return states, policies, values

def train_wyvern(model, epochs=10, games_per_epoch=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    model.train()

    print("🚀 Self-Play 강화학습 시작...")
    for epoch in range(1, epochs + 1):
        all_states, all_policies, all_values = [], [], []

        for _ in range(games_per_epoch):
            s, p, v = self_play_game(model, device)
            all_states.extend(s)
            all_policies.extend(p)
            all_values.extend(v)

        X = torch.stack(all_states).to(device)
        Y_policy = torch.tensor(np.array(all_policies), dtype=torch.float32).to(device)
        Y_value = torch.tensor(np.array(all_values), dtype=torch.float32).to(device)

        optimizer.zero_grad()
        p_pred, v_pred = model(X)

        loss_p = F.cross_entropy(p_pred, Y_policy)
        loss_v = F.mse_loss(v_pred, Y_value)
        total_loss = loss_p + loss_v

        total_loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss.item():.4f} (Policy: {loss_p.item():.4f}, Value: {loss_v.item():.4f})")

        # 5 에포크마다 가중치 구글 드라이브 백업
        if epoch % 5 == 0:
            torch.save(model.state_dict(), f"{SAVE_DIR}/wyvern_ckpt_epoch_{epoch}.pt")

# 실행 (필요에 따라 epoch와 games_per_epoch 숫자를 늘려보세요!)
train_wyvern(model, epochs=10, games_per_epoch=15)

🚀 Self-Play 강화학습 시작...
Epoch [1/10] - Loss: 7.6626 (Policy: 7.5853, Value: 0.0773)
Epoch [2/10] - Loss: 6.8033 (Policy: 6.7928, Value: 0.0105)
Epoch [3/10] - Loss: 818.1313 (Policy: 818.1219, Value: 0.0094)
Epoch [4/10] - Loss: 6.6386 (Policy: 6.6302, Value: 0.0084)
Epoch [5/10] - Loss: 7.0258 (Policy: 7.0184, Value: 0.0074)
Epoch [6/10] - Loss: 7.3390 (Policy: 7.3325, Value: 0.0065)
Epoch [7/10] - Loss: 7.3198 (Policy: 7.3142, Value: 0.0057)
Epoch [8/10] - Loss: 7.2296 (Policy: 7.1889, Value: 0.0407)
Epoch [9/10] - Loss: 6.7994 (Policy: 6.7952, Value: 0.0042)
Epoch [10/10] - Loss: 5.7861 (Policy: 5.7159, Value: 0.0702)


In [4]:
# Colab Cell 4 - 레벨별 와이번 체크포인트 내보내기 함수
import os, torch, onnx

def export_wyvern_level(model, level_name, device):
    """
    예시 usage:
    export_wyvern_level(model, "wyvern_candidate_elo2000", device)
    """
    model.eval()

    # 1. PyTorch 원본 체크포인트 저장 (.pt)
    pt_path = f"{SAVE_DIR}/{level_name}.pt"
    torch.save(model.state_dict(), pt_path)

    # 2. 25MB/50MB/100MB 등 Java 연동용 ONNX 내보내기 (.onnx)
    dummy_input = torch.randn(1, 12, 8, 8, device=device)
    onnx_path = f"{SAVE_DIR}/{level_name}.onnx"

    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['board_input'],
        output_names=['policy_output', 'value_output'],
        dynamic_axes={'board_input': {0: 'batch_size'}}
    )

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"🐉 [Wyvern Level-Up!] {level_name} 변환 완료!")
    print(f"📁 저장 경로: {onnx_path}")
    print(f"⚖️ 파라미터 용량: {size_mb:.2f} MB\n")

# 사용 예시: ELO 2000 궤도 진입 시 호출
# export_wyvern_level(model, "wyvern_candidate", device)